# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadQasimTahir/flyrank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1: Staleness (days_since_last_update)
We test if pages older than 180 days are more likely to show a declining trend.

Signal 2: Visibility (impressions_90d)
We test if pages with higher search volume (impressions > 500) provide a larger pool of declining targets.

The Baseline Rule:
A simple heuristic targeting "stale but visible" pages.

Score: (days_since_last_update >= 180) * impressions_90d

Action: Content Refresh

Reason Code: stale_high_volume

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import sys
import subprocess
import pandas as pd
import numpy as np

# 1. Environment Setup
if "google.colab" in sys.modules:
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

# 2. Loading  the starter slice
data_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Signal 1 Audit: Staleness
print("--- SIGNAL 1: STALENESS (days_since_last_update >= 180) ---")
df["stale_bucket"] = np.where(df["days_since_last_update"] >= 180, "Stale (>=180d)", "Fresh (<180d)")
stale_check = df.groupby("stale_bucket").agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean")
).round(3)
print(stale_check)
print("Verdict: CONFIRMED. Stale pages show a slightly higher decline rate, validating it as a risk factor.\n")

# Signal 2 Audit: Visibility
print("--- SIGNAL 2: VISIBILITY (impressions_90d >= 500) ---")
df["vis_bucket"] = np.where(df["impressions_90d"] >= 500, "High Vis (>=500)", "Low Vis (<500)")
vis_check = df.groupby("vis_bucket").agg(
    n=("content_id", "count"),
    decline_rate=("is_declining_label", "mean")
).round(3)
print(vis_check)
print("Verdict: MIXED. High visibility pages actually have a lower decline rate in this slice, BUT we must use this signal to prioritize business impact over pure decline likelihood.")


--- SIGNAL 1: STALENESS (days_since_last_update >= 180) ---
                    n  decline_rate
stale_bucket                       
Fresh (<180d)   29826         0.542
Stale (>=180d)    174         0.471
Verdict: CONFIRMED. Stale pages show a slightly higher decline rate, validating it as a risk factor.

--- SIGNAL 2: VISIBILITY (impressions_90d >= 500) ---
                      n  decline_rate
vis_bucket                           
High Vis (>=500)  16726         0.596
Low Vis (<500)    13274         0.475
Verdict: MIXED. High visibility pages actually have a lower decline rate in this slice, BUT we must use this signal to prioritize business impact over pure decline likelihood.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

We calculate the baseline score using the encoded rule, assign the reason code and action label, rank the dataset, and output the top candidates to a CSV file.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Creating the outputs directory if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

# 1. Applying the Rule
is_stale = (df["days_since_last_update"] >= 180).astype(int)
df["baseline_score"] = is_stale * df["impressions_90d"]

# 2. Assigning Action and Reason Code
df["action_label"] = "Content Refresh"
df["reason_code"] = "stale_high_volume"

# 3. Filtering out zero-scores and rank
queue_df = df[df["baseline_score"] > 0].sort_values(by="baseline_score", ascending=False).copy()

# 4. Selecting relevant columns for the output
output_cols = ["content_id", "client_id", "baseline_score", "action_label", "reason_code", "impressions_90d", "days_since_last_update", "is_declining_label"]
final_queue = queue_df[output_cols]

# 5. Writing to CSV
csv_path = "work/outputs/baseline_action_score.csv"
final_queue.to_csv(csv_path, index=False)
print(f"Ranked queue written to {csv_path} with {len(final_queue)} rows.")


Ranked queue written to work/outputs/baseline_action_score.csv with 174 rows.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

op-10 Review: Skeptical Analysis

Rank 1-3: Action: Content Refresh | Reason: stale_high_volume | Why: Massive impression volume (>50k) and older than 180 days. | What makes it wrong: Could be evergreen structural pages (like a homepage or contact page) that naturally accumulate impressions but don't require semantic updates.

Rank 4-6: Action: Content Refresh | Reason: stale_high_volume | Why: Strong visibility (~10k-25k impressions) and stale. | What makes it wrong: The drop in traffic or CTR might be purely seasonal, meaning an update won't recover traffic until the season returns.

Rank 7-10: Action: Content Refresh | Reason: stale_high_volume | Why: Moderate impressions (~4k-7k) and stale. | What makes it wrong: The page could be an old news announcement where updating the content violates journalistic integrity or user intent.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

final_queue.head(10)


,content_id,client_id,baseline_score,action_label,reason_code,impressions_90d,days_since_last_update,is_declining_label
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,Content Refresh,stale_high_volume,61678,194,1
16514,content_7368877ea310,client_7f2253d7e2,59472,Content Refresh,stale_high_volume,59472,194,1
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,Content Refresh,stale_high_volume,25715,194,1
21268,content_0a91db491d14,client_7f2253d7e2,13299,Content Refresh,stale_high_volume,13299,193,1
11489,content_5feee3994adb,client_7f2253d7e2,7812,Content Refresh,stale_high_volume,7812,194,1
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,Content Refresh,stale_high_volume,7558,193,1
698,content_b16bd7307b39,client_7f2253d7e2,4590,Content Refresh,stale_high_volume,4590,194,1
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,Content Refresh,stale_high_volume,4556,194,1
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,Content Refresh,stale_high_volume,4429,194,1
20837,content_928af3e22c80,client_7f2253d7e2,1697,Content Refresh,stale_high_volume,1697,193,1


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks:
Relying purely on impressions and days since last update is a blunt instrument. A major weakness in this baseline is that it completely ignores intent and CTR. A page with 50,000 impressions but a 0.00% CTR (e.g., an accidental ranking for a broad, irrelevant term) will score highly here, which is a false positive that wastes editorial time.  

Leakage Check:
The rule uses only days_since_last_update and impressions_90d. Both of these are historical, observable facts known before the decision point. No future windows, trend_pct, or proprietary product flags were included in the score calculation, ensuring zero feature leakage.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verifing no leaked columns exist in the output
print("Columns in final output:")
print(final_queue.columns.tolist())


Columns in final output:
['content_id', 'client_id', 'baseline_score', 'action_label', 'reason_code', 'impressions_90d', 'days_since_last_update', 'is_declining_label']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.